# Week06 — 쇼핑몰 리뷰 감성분석 (Colab 완성본)
합성 리뷰 120건, 규칙 기반과 AI 모델 결과 비교. 실제 리뷰가 아닙니다.


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report
csv_url='https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/Week06/data/shop_reviews_synthetic_120.csv'
df=pd.read_csv(csv_url,encoding='utf-8-sig')
print(len(df)); display(df.head())


## 실습 1: 데이터 정제


In [ ]:
df=df.drop_duplicates('review_id').dropna(subset=['review_text']).copy()
df['review_text']=df['review_text'].astype(str).str.strip().str.replace(r'\s+',' ',regex=True)
print(df.isna().sum());print('리뷰 건수',len(df))


## 실습 2: 수작업 키워드 규칙
먼저 실제 문장 10건을 사람이 직접 읽고 분류한 다음, 아래 규칙과 비교합니다.


In [ ]:
positive=['좋아','만족','추천','빠르','꼼꼼','편리']
negative=['늦','불편','불량','실망','고장','별로']
def rule(t):
    p=sum(x in t for x in positive)
    n=sum(x in t for x in negative)
    return '긍정' if p>n else '부정' if n>p else '중립'
df['manual_rule']=df.review_text.map(rule)
display(df[['review_text','manual_rule']].head(10))


## 실습 3: 사전학습 AI 감성분석
금융 텍스트용 snunlp/KR-FinBert-SC를 사용합니다. 쇼핑몰 도메인과 다르므로 결과를 무조건 신뢰하지 마세요.


In [ ]:
!pip -q install transformers torch
from transformers import pipeline
clf=pipeline('text-classification',model='snunlp/KR-FinBert-SC',device=-1)
print('모델 라벨',clf.model.config.id2label)
preds=clf(df.review_text.tolist(),batch_size=8,truncation=True,max_length=256)
print(preds[:3])


In [ ]:
mapping={'positive':'긍정','negative':'부정','neutral':'중립','긍정':'긍정','부정':'부정','중립':'중립'}
def to_kr(label):
    label=label.lower().strip()
    if label not in mapping: raise ValueError('라벨 확인 필요: '+label)
    return mapping[label]
df['ai_sentiment']=[to_kr(x['label']) for x in preds]
df['ai_score']=[x['score'] for x in preds]
display(df[['review_text','manual_rule','ai_sentiment','ai_score']].head(10))


## 실습 4: 결과 비교·불일치 분석


In [ ]:
print('규칙-AI 일치율',(df.manual_rule==df.ai_sentiment).mean())
display(pd.crosstab(df.manual_rule,df.ai_sentiment))
print('교육용 참고 라벨에 대한 규칙')
print(classification_report(df.reference_sentiment,df.manual_rule,zero_division=0))
print('교육용 참고 라벨에 대한 AI')
print(classification_report(df.reference_sentiment,df.ai_sentiment,zero_division=0))
diff=df.loc[df.manual_rule!=df.ai_sentiment]
display(diff.head(20))
df.to_csv('week06_results.csv',index=False,encoding='utf-8-sig')
diff.to_csv('week06_disagreements.csv',index=False,encoding='utf-8-sig')


## 강의자 메모
모델 결과는 실시간 실행 검증 전입니다. 금융 도메인 모델의 쇼핑 리뷰 오분류 사례를 찾도록 지도합니다. 참고 라벨은 합성이며 객관적인 정답이 아닙니다.
